# News Domain Link Graph Analysis

Weighted link counts between ~1,200 news domains extracted from Common Crawl CC-MAIN-2024-38.

Source: `count_news_links.py` job on AWS EMR Serverless — fetched 2.9M WARC records, parsed HTML `<a href>` links, aggregated per (source_domain, target_domain) pair.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import glob

files = glob.glob('../results/news_links/part-*.parquet')
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
print(f"Loaded {len(df):,} rows from {len(files)} parquet files")

## Overview

In [ ]:
print(f"Unique (source, target) pairs: {len(df):,}")
print(f"Unique source domains:          {df['s'].nunique():,}")
print(f"Unique target domains:          {df['t'].nunique():,}")
print(f"Total link occurrences (sum):   {df['count'].sum():,}")
print()
print("Count distribution:")
print(df['count'].describe().apply(lambda x: f"{x:,.1f}"))

## Link Count Distribution: Mean vs Median

The distribution is extremely right-skewed. The **median edge weight is 2** while the **mean is 183** — a 90× gap driven by a small number of very high-count edges (site-wide navigation/footer links) inflating the mean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: log-scale histogram of raw counts
ax = axes[0]
ax.hist(df['count'], bins=100, color='steelblue', edgecolor='none')
ax.set_yscale('log')
ax.set_xscale('log')
ax.axvline(df['count'].median(), color='orange', linestyle='--', label=f"Median = {df['count'].median():.0f}")
ax.axvline(df['count'].mean(),   color='red',    linestyle='--', label=f"Mean   = {df['count'].mean():.0f}")
ax.set_xlabel('Link count (log scale)')
ax.set_ylabel('Number of edges (log scale)')
ax.set_title('Edge weight distribution (log-log)')
ax.legend()

# Right: cumulative % of total link volume by count threshold
ax2 = axes[1]
sorted_counts = df['count'].sort_values(ascending=False).values
cumulative = np.cumsum(sorted_counts) / sorted_counts.sum() * 100
ax2.plot(range(1, len(cumulative)+1), cumulative, color='steelblue')
ax2.set_xscale('log')
ax2.set_xlabel('Top-N edges (log scale)')
ax2.set_ylabel('% of total link volume')
ax2.set_title('Cumulative link volume by top-N edges')
ax2.axhline(50, color='orange', linestyle='--', label='50%')
ax2.axhline(90, color='red',    linestyle='--', label='90%')
ax2.legend()

plt.tight_layout()
plt.show()

top1pct = int(len(df) * 0.01)
vol_top1pct = sorted_counts[:top1pct].sum() / sorted_counts.sum() * 100
print(f"Top 1% of edges ({top1pct:,}) account for {vol_top1pct:.1f}% of total link volume")

## Top Domains

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_in  = df.groupby('t')['count'].sum().sort_values(ascending=False).head(20)
top_out = df.groupby('s')['count'].sum().sort_values(ascending=False).head(20)

axes[0].barh(top_in.index[::-1], top_in.values[::-1], color='steelblue')
axes[0].set_title('Top 20 most-linked-to domains (incoming link volume)')
axes[0].set_xlabel('Total incoming link count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))

axes[1].barh(top_out.index[::-1], top_out.values[::-1], color='darkorange')
axes[1].set_title('Top 20 highest-linking-out domains (outgoing link volume)')
axes[1].set_xlabel('Total outgoing link count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x>=1e6 else f'{x/1e3:.0f}K'))

plt.tight_layout()
plt.show()

## Footer/Nav Links vs Editorial Links

High link counts between two domains almost certainly reflect **site-wide footer or navigation bars** — a link repeated on every page of a site. A single nav link on a 250K-page site produces a count of 250K for one (s, t) pair.

True **editorial links** — a journalist citing another outlet — appear rarely and produce low counts (typically 1–10 per pair).

We split the data using a threshold of **100 links per pair**: above = likely nav/footer, below = likely editorial.

In [ ]:
NAV_THRESHOLD = 100

nav = df[df['count'] >= NAV_THRESHOLD]
editorial = df[df['count'] < NAV_THRESHOLD]

print(f"Nav/footer edges (count >= {NAV_THRESHOLD}): {len(nav):,}  ({len(nav)/len(df)*100:.1f}% of pairs, "
      f"{nav['count'].sum()/df['count'].sum()*100:.1f}% of link volume)")
print(f"Editorial edges  (count <  {NAV_THRESHOLD}): {len(editorial):,}  ({len(editorial)/len(df)*100:.1f}% of pairs, "
      f"{editorial['count'].sum()/df['count'].sum()*100:.1f}% of link volume)")
print()
print("=== Top nav/footer edges ===")
print(nav.nlargest(15, 'count')[['s', 't', 'count']].to_string(index=False))
print()
print("=== Top editorial edges (count 10–99, showing most frequent cross-outlet citations) ===")
ed_top = editorial[editorial['count'] >= 10].nlargest(15, 'count')
print(ed_top[['s', 't', 'count']].to_string(index=False))

## Network Groups: Reach PLC and Fox

Two network effects dominate the high-count edges.

In [ ]:
# Reach PLC titles: UK regional papers owned by Reach PLC that cross-link heavily to mirror.co.uk
reach_titles = [
    'mirror.co.uk', 'birminghammail.co.uk', 'walesonline.co.uk',
    'manchestereveningnews.co.uk', 'chroniclelive.co.uk', 'hulldailymail.co.uk',
    'examinerlive.co.uk', 'liverpoolecho.co.uk', 'mylondon.news', 'devonlive.com',
    'cheshire-live.co.uk', 'nottinghampost.com', 'getsurrey.co.uk',
    'leicestermercury.co.uk', 'stokesentinel.co.uk', 'dailyrecord.co.uk',
    'dailystar.co.uk',
]

# Fox network affiliates
fox_titles = [d for d in df['s'].unique() if 'fox' in d or d in ('cozitv.com', 'my9nj.com')]

reach_internal = df[df['s'].isin(reach_titles) & df['t'].isin(reach_titles)]
fox_internal   = df[df['s'].isin(fox_titles)   & df['t'].isin(fox_titles)]
other          = df[~df['s'].isin(reach_titles + fox_titles) | ~df['t'].isin(reach_titles + fox_titles)]

print("=== Reach PLC internal link volume ===")
print(f"  Edges: {len(reach_internal):,}  |  Link volume: {reach_internal['count'].sum():,}")
print(reach_internal.nlargest(8, 'count')[['s','t','count']].to_string(index=False))
print()
print("=== Fox network internal link volume ===")
print(f"  Edges: {len(fox_internal):,}  |  Link volume: {fox_internal['count'].sum():,}")
print(fox_internal.nlargest(8, 'count')[['s','t','count']].to_string(index=False))

## Editorial-Only Graph: Top Targets and Sources

Filtering to editorial edges (count < 100) removes the nav/footer noise and reveals genuine cross-outlet citations.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ed_in  = editorial.groupby('t')['count'].sum().sort_values(ascending=False).head(20)
ed_out = editorial.groupby('s')['count'].sum().sort_values(ascending=False).head(20)

axes[0].barh(ed_in.index[::-1], ed_in.values[::-1], color='steelblue')
axes[0].set_title('Top 20 most-cited targets (editorial only, count < 100)')
axes[0].set_xlabel('Total editorial incoming links')

axes[1].barh(ed_out.index[::-1], ed_out.values[::-1], color='darkorange')
axes[1].set_title('Top 20 most-citing sources (editorial only, count < 100)')
axes[1].set_xlabel('Total editorial outgoing links')

plt.tight_layout()
plt.show()

## Log-Scaled Edge Weights

For downstream graph analysis, raw counts are dominated by nav/footer links. Two common normalisations:

- **log1p(count)** — compresses the range, brings nav/footer edges down without discarding them
- **Threshold clip** — cap at e.g. 100, treating all nav-like edges equally

In [ ]:
df['weight_log']  = np.log1p(df['count'])
df['weight_clip'] = df['count'].clip(upper=NAV_THRESHOLD)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['weight_log'], bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(df['weight_log'].median(), color='orange', linestyle='--',
                label=f"Median = {df['weight_log'].median():.2f}")
axes[0].axvline(df['weight_log'].mean(), color='red', linestyle='--',
                label=f"Mean   = {df['weight_log'].mean():.2f}")
axes[0].set_xlabel('log1p(count)')
axes[0].set_ylabel('Number of edges')
axes[0].set_title('log1p-scaled weights')
axes[0].legend()

axes[1].hist(df['weight_clip'], bins=50, color='darkorange', edgecolor='none')
axes[1].axvline(df['weight_clip'].median(), color='blue', linestyle='--',
                label=f"Median = {df['weight_clip'].median():.0f}")
axes[1].axvline(df['weight_clip'].mean(), color='red', linestyle='--',
                label=f"Mean   = {df['weight_clip'].mean():.1f}")
axes[1].set_xlabel(f'count clipped at {NAV_THRESHOLD}')
axes[1].set_ylabel('Number of edges')
axes[1].set_title(f'Clipped weights (cap = {NAV_THRESHOLD})')
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| (source, target) pairs | 21,788 |
| Unique sources | 376 |
| Unique targets | 1,043 |
| Total link occurrences | ~4M |
| Median edge weight | 2 |
| Mean edge weight | 183 |
| Nav/footer edges (≥100) | ~4% of pairs, ~97% of volume |
| Editorial edges (<100) | ~96% of pairs, ~3% of volume |

**Key findings:**
- The graph is dominated by two media networks: **Reach PLC** (UK regional papers all cross-linking to `mirror.co.uk` via site-wide nav) and the **Fox affiliate network** (US local TV stations cross-linking to each other).
- These nav/footer edges account for the vast majority of raw link volume but carry little editorial signal.
- For graph analysis, use **editorial-only edges** (count < 100) or **log1p weights** to surface genuine cross-outlet citation patterns.